# 📉 Lean Six Sigma — Phase 5: Control
## Databricks Monitoring Pipeline — Real-Time SLA Tracking

This notebook simulates the Databricks Delta Lake pipeline that ingests daily CRM exports and surfaces SLA breach alerts in Tableau.

**Architecture:**
```
Salesforce CRM (daily export)
    → Databricks Delta: proposals_raw
    → Databricks Delta: proposal_kpis (cycle time, sigma, SLA status)
    → Tableau: real-time SLA monitoring dashboard
    → Email alert when 7-day avg > 30 days
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../data/proposals.csv', parse_dates=['submission_date'])
print('Pipeline simulation initialized')
print(f'Total records: {len(df)}')

In [ ]:
# ── DATABRICKS DELTA LAYER SIMULATION ──────────────────────────────────────
# In production this runs as a Databricks job on a schedule
# spark.readStream.format('delta').load('/mnt/crm/proposals_raw')
# Here we simulate with pandas for portability

def compute_kpis(df_chunk):
    """Mirrors the Databricks Delta transform logic for proposal_kpis table."""
    kpis = {
        'avg_cycle_time': df_chunk['cycle_time_days'].mean(),
        'sla_breach_rate': df_chunk['sla_breach'].mean() * 100,
        'rework_rate': df_chunk['rework_flag'].mean() * 100,
        'p95_cycle_time': df_chunk['cycle_time_days'].quantile(0.95),
        'total_proposals': len(df_chunk),
        'sigma_level': round(stats.norm.ppf(1 - df_chunk['sla_breach'].mean()) + 1.5, 2) if df_chunk['sla_breach'].mean() < 1 else 0
    }
    return kpis

# Simulate rolling 7-day window KPI tracking
df_sorted = df.sort_values('submission_date').reset_index(drop=True)
window_kpis = []

for i in range(14, len(df_sorted), 3):
    window = df_sorted.iloc[max(0, i-14):i]
    kpi = compute_kpis(window)
    kpi['date'] = df_sorted.iloc[i]['submission_date']
    window_kpis.append(kpi)

kpi_df = pd.DataFrame(window_kpis)
print('=== proposal_kpis Delta Table (sample) ===')
print(kpi_df[['date','avg_cycle_time','sla_breach_rate','sigma_level']].tail(10).to_string(index=False))

In [ ]:
# Alert logic — mirrors Databricks scheduled alert
ALERT_THRESHOLD = 30  # days
alerts = kpi_df[kpi_df['avg_cycle_time'] > ALERT_THRESHOLD]
print(f'=== SLA ALERTS TRIGGERED: {len(alerts)} ===' )
if len(alerts) > 0:
    print(alerts[['date','avg_cycle_time','sla_breach_rate']].to_string(index=False))
else:
    print('No alerts — all windows within SLA target')

In [ ]:
# Tableau-ready dashboard output
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Databricks → Tableau Live SLA Monitoring Dashboard', fontsize=14, fontweight='bold')

# Panel 1: Avg cycle time over time with alert threshold
axes[0,0].plot(kpi_df['date'], kpi_df['avg_cycle_time'], color='#378ADD', linewidth=2)
axes[0,0].fill_between(kpi_df['date'], kpi_df['avg_cycle_time'], alpha=0.15, color='#378ADD')
axes[0,0].axhline(ALERT_THRESHOLD, color='#E24B4A', linestyle='--', linewidth=1.5, label=f'Alert Threshold ({ALERT_THRESHOLD}d)')
axes[0,0].axhline(35, color='black', linestyle=':', linewidth=1.5, label='SLA Limit (35d)')
axes[0,0].xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
axes[0,0].set_title('7-Day Rolling Avg Cycle Time')
axes[0,0].set_ylabel('Days')
axes[0,0].legend(fontsize=9)

# Panel 2: SLA breach rate
axes[0,1].bar(kpi_df['date'], kpi_df['sla_breach_rate'],
              color=['#E24B4A' if v > 20 else '#5DCAA5' for v in kpi_df['sla_breach_rate']],
              width=3, alpha=0.85)
axes[0,1].xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
axes[0,1].set_title('SLA Breach Rate % (7-day window)')
axes[0,1].set_ylabel('%')

# Panel 3: Sigma level trend
valid = kpi_df[kpi_df['sigma_level'] > 0]
axes[1,0].plot(valid['date'], valid['sigma_level'], color='#1D9E75', linewidth=2, marker='o', markersize=3)
axes[1,0].axhline(3.0, color='#BA7517', linestyle='--', linewidth=1.5, label='Target: 3σ')
axes[1,0].xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
axes[1,0].set_title('Process Sigma Level Over Time')
axes[1,0].set_ylabel('Sigma Level')
axes[1,0].legend()

# Panel 4: Summary scorecard
latest = kpi_df.iloc[-1]
early = kpi_df.iloc[5]
scorecard = {
    'Avg Cycle Time': (f"{early['avg_cycle_time']:.1f}d", f"{latest['avg_cycle_time']:.1f}d"),
    'SLA Breach Rate': (f"{early['sla_breach_rate']:.1f}%", f"{latest['sla_breach_rate']:.1f}%"),
    'Sigma Level': (f"{early['sigma_level']:.2f}", f"{latest['sigma_level']:.2f}"),
    'P95 Cycle Time': (f"{early['p95_cycle_time']:.1f}d", f"{latest['p95_cycle_time']:.1f}d")
}
axes[1,1].axis('off')
table_data = [[k, v[0], v[1]] for k, v in scorecard.items()]
table = axes[1,1].table(cellText=table_data, colLabels=['KPI', 'Baseline', 'Current'],
                         cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2)
axes[1,1].set_title('KPI Scorecard', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../outputs/databricks_tableau_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved — ready for Tableau import via CSV export')
kpi_df.to_csv('../outputs/proposal_kpis_export.csv', index=False)
print('KPI table exported: outputs/proposal_kpis_export.csv')